In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import IntEnum
from typing import Any, Protocol

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from utils.config import RuleSet
from utils.scoring import legal_keeps, score_counts  


class Decision(IntEnum):
    ROLL = 0
    BANK = 1


class OpponentPolicy(Protocol):
    def play_turn(
        self, rules: RuleSet, rng: np.random.Generator,
        own_score: int, best_rival_score: int, on_board: bool,
    ) -> int:
        """Play one full turn; return points actually banked (0 on farkle)."""

In [ ]:
@dataclass(frozen=True)
class ThresholdOpponent:
    """Baseline opponent: take the best scoring keep and bank at a threshold."""

    bank_at: int = 500

    def play_turn(
        self, rules: RuleSet, rng: np.random.Generator,
        own_score: int, best_rival_score: int, on_board: bool,
    ) -> int:
        dice_left, turn_score = rules.num_dice, 0
        while True:
            counts = np.bincount(rng.integers(1, 7, size=dice_left) - 1,
                                 minlength=6)
            keeps = legal_keeps(counts, rules)
            if not keeps:
                return 0
            dice_used, (score, _) = max(
                keeps.items(), key=lambda item: (item[1][0], item[0])
            )
            turn_score += score
            dice_left -= dice_used
            can_bank = on_board or turn_score >= rules.min_first_bank
            if can_bank and (
                turn_score >= self.bank_at
                or own_score + turn_score >= rules.target_score
                or (dice_left == 0 and not rules.hot_dice)
            ):
                return turn_score
            if dice_left == 0:
                if not rules.hot_dice:
                    return 0  # Cannot roll or meet the entry threshold.
                dice_left = rules.num_dice


In [ ]:
class FarkleEnv(gym.Env):
    """Farkle from one seat. Opponent turns are simulated inside `step`.

    Action space is flat: 2 * num_dice actions, decoded as
    (dice_kept, Decision). `dice_kept` selects the *highest-scoring* keep
    that uses exactly that many dice -- lower-scoring keeps of the same
    size are provably dominated, since the turn's future depends only on
    (dice_left, turn_score).
    """

    metadata = {"render_modes": ["human", "ansi"]}

    def __init__(
        self,
        rules: RuleSet | str = "configs/rules/standard_10k.yaml",
        opponent: OpponentPolicy | None = None,
        reward_mode: str = "points",       # "points" | "win" | "both"
        max_turns: int = 50,
        render_mode: str | None = None,
    ):
        super().__init__()
        self.rules = rules if isinstance(rules, RuleSet) else RuleSet.from_yaml(rules)
        self.rules.validate()
        self.opponent = opponent if opponent is not None else ThresholdOpponent()
        self.reward_mode = reward_mode
        self.max_turns = max_turns
        self.render_mode = render_mode

        n = self.rules.num_dice
        self.action_space = spaces.Discrete(2 * n)
        self.observation_space = spaces.Dict({
            "roll": spaces.MultiDiscrete([n + 1] * 6),
            "dice_left": spaces.Discrete(n + 1),
            "turn_score": spaces.Box(0.0, np.inf, shape=(1,), dtype=np.float32),
            "my_score": spaces.Box(0.0, np.inf, shape=(1,), dtype=np.float32),
            "opp_score": spaces.Box(0.0, np.inf, shape=(1,), dtype=np.float32),
            "on_board": spaces.Discrete(2),
            "opp_on_board": spaces.Discrete(2),
            "final_round": spaces.Discrete(2),
            "finisher": spaces.Discrete(3),  # 0=none, 1=me, 2=opponent
            "turns_remaining": spaces.Discrete(max_turns + 1),
        })
        self._terminated = False
        self._truncated = False
        self._counts = np.zeros(6, dtype=np.int64)
        self._keeps = {}

    def _decode(self, action: int) -> tuple[int, Decision]:
        return action // 2 + 1, Decision(action % 2)

    def _encode(self, dice_kept: int, decision: Decision) -> int:
        return (dice_kept - 1) * 2 + int(decision)

    def action_mask(self) -> np.ndarray:
        mask = np.zeros(self.action_space.n, dtype=bool)
        if self._terminated or self._truncated:
            return mask
        if not self._keeps:
            mask[0] = True  # A farkle on the opening roll: forced pass.
            return mask
        for dice_used, (score, _) in self._keeps.items():
            remaining = self._dice_left - dice_used
            if remaining > 0 or self.rules.hot_dice:
                mask[self._encode(dice_used, Decision.ROLL)] = True
            if (self._bank_allowed(self._turn_score + score)
                    or (remaining == 0 and not self.rules.hot_dice)):
                # On non-hot dice, below-threshold banking can be a forced
                # zero-point turn when no more dice remain.
                mask[self._encode(dice_used, Decision.BANK)] = True
        return mask

    def _bank_allowed(self, prospective_total: int) -> bool:
        """Banking below the first-bank threshold scores nothing and ends the
        turn -- strictly dominated by rolling, since the turn score at risk is
        already worth zero. Mask it out."""
        if self._on_board:
            return True
        return prospective_total >= self.rules.min_first_bank


    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._my_score = 0
        self._opp_score = 0
        self._on_board = False
        self._opp_on_board = False
        self._turn_count = 0
        self._final_round = False
        self._finisher: str | None = None
        self._terminated = False
        self._truncated = False
        self._begin_turn()
        return self._obs(), self._info()

    def step(self, action: int):
        if self._terminated or self._truncated:
            raise RuntimeError("episode finished; call reset()")
        if not self.action_space.contains(action):
            raise ValueError(f"action outside action space: {action}")
        action = int(action)
        if not self.action_mask()[action]:
            raise ValueError(f"illegal action {action}")

        banked = 0
        if not self._keeps:
            # The opening roll farkled. Action 0 consumes this turn.
            self._end_turn()
        else:
            dice_kept, decision = self._decode(action)
            score, _ = self._keeps[dice_kept]
            self._turn_score += score
            self._dice_left -= dice_kept
            if decision is Decision.BANK:
                banked = self._bank()
                self._end_turn()
            else:
                if self._dice_left == 0:  # Hot dice; ROLL is otherwise masked.
                    self._dice_left = self.rules.num_dice
                self._roll()
                if not self._keeps:
                    self._turn_score = 0
                    self._end_turn()

        reward = self._reward(banked, self._terminated)
        if self.render_mode == "human":
            self.render()
        return self._obs(), reward, self._terminated, self._truncated, self._info()



    def _roll(self) -> None:
        """Roll self._dice_left dice into self._counts; recompute self._keeps."""
        rolled = np.random.choice(range(1,7),self._dice_left)
        self._counts = np.bincount(rolled - 1, minlength=6)
        self._keeps = legal_keeps(self._counts, self.rules)
        

    def _begin_turn(self) -> None:
        """Reset turn score, dice_left = num_dice, roll. NB: the opening roll
        can itself be a farkle -- handle a zero-length turn."""
        self._turn_score = 0
        self._dice_left = self.rules.num_dice
        self._roll()


    def _bank(self) -> int:
        if not self._bank_allowed(self._turn_score):
            return 0
        banked = self._turn_score
        self._my_score += banked
        self._on_board = True
        if self._finisher is None and self._my_score >= self.rules.target_score:
            self._finisher = "me"
            self._final_round = bool(self.rules.final_round)
        return banked

    def _end_turn(self) -> None:
        """Resolve the opponent and the two possible final-round orders."""
        self._turn_count += 1
        # If the opponent crossed on its preceding turn, this was my reply.
        if self._finisher == "opp":
            self._terminated = True
            return
        if self._finisher == "me" and not self.rules.final_round:
            self._terminated = True
            return

        gained = self.opponent.play_turn(
            self.rules, self.np_random, self._opp_score,
            self._my_score, self._opp_on_board,
        )
        if not isinstance(gained, (int, np.integer)) or gained < 0:
            raise ValueError("opponent must return nonnegative integer banked points")
        if gained:
            if not self._opp_on_board and gained < self.rules.min_first_bank:
                raise ValueError("opponent banked below its entry threshold")
            self._opp_score += int(gained)
            self._opp_on_board = True
            if self._finisher is None and self._opp_score >= self.rules.target_score:
                self._finisher = "opp"
                self._final_round = bool(self.rules.final_round)

        # If I reached the target first, the opponent just took its last turn.
        if self._finisher == "me" or (
            self._finisher == "opp" and not self.rules.final_round
        ):
            self._terminated = True
        elif self._turn_count >= self.max_turns:
            self._truncated = True
        else:
            self._begin_turn()

    def _reward(self, banked: int, terminated: bool) -> float:
        point_reward = banked / self.rules.target_score
        outcome_reward = (float(self._my_score > self._opp_score)
                          - float(self._my_score < self._opp_score)) if terminated else 0.0
        if self.reward_mode == "points":
            return point_reward
        if self.reward_mode == "win":
            return outcome_reward
        return point_reward + outcome_reward

    def _obs(self) -> dict[str, Any]:
        target = float(self.rules.target_score)
        return {
            "roll": self._counts.copy(),
            "dice_left": self._dice_left,
            "turn_score": np.array([self._turn_score / target], dtype=np.float32),
            "my_score": np.array([self._my_score / target], dtype=np.float32),
            "opp_score": np.array([self._opp_score / target], dtype=np.float32),
            "on_board": int(self._on_board),
            "opp_on_board": int(self._opp_on_board),
            "final_round": int(self._final_round),
            "finisher": {None: 0, "me": 1, "opp": 2}[self._finisher],
            "turns_remaining": max(0, self.max_turns - self._turn_count),
        }

    def _info(self) -> dict[str, Any]:
        return {
            "action_mask": self.action_mask(),
            "turn_score": self._turn_score,
            "my_score": self._my_score,
            "opp_score": self._opp_score,
            "turn_count": self._turn_count,
            "winner": ("me" if self._my_score > self._opp_score else
                       "opp" if self._opp_score > self._my_score else "tie")
                       if self._terminated else None,
            "rules_fingerprint": self.rules.fingerprint(),
        }

    def render(self):
        if self.render_mode is None:
            return None
        description = (f"roll={self._counts.tolist()} left={self._dice_left} "
                       f"turn={self._turn_score} scores={self._my_score}:"
                       f"{self._opp_score} finisher={self._finisher}")
        if self.render_mode == "human":
            print(description)
            return None
        return description